# 📓 Semana 6 · Dia 4 — CI/CD com GitHub Actions + DABs

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | 🔑 Versão paga (deploy) — CI parcial na Free |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (CI/CD) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline CI/CD rodando no GitHub |

---


## 📖 Teoria — CI vs CD

**CI (Continuous Integration)**: ao fazer push/PR, roda validação — `bundle validate`, testes, lint. Pega erro cedo.
**CD (Continuous Deployment)**: após o merge na main, faz `bundle deploy -t prod` automaticamente.

Os **blueprints oficiais** do Databricks (GitHub Actions) já trazem esse padrão pronto.


### 💻 Na prática — GitHub Actions para CI (roda localmente/validate — funciona na Free)

Crie `.github/workflows/ci.yml` no repositório:


In [ ]:
# .github/workflows/ci.yml
yaml_ci = """
name: CI
on:
  pull_request:
  push:
    branches: [main]
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - name: Validate bundle
        run: databricks bundle validate -t dev
"""
print(yaml_ci)
print("Coloque em .github/workflows/ci.yml e faça push.")

### 💻 Na prática — CD no merge

O workflow de CD roda o deploy no ambiente prod após merge na main.


In [ ]:
# .github/workflows/cd.yml (usar secrets DATABRICKS_HOST/TOKEN no repo)
yaml_cd = """
name: CD
on:
  push:
    branches: [main]
jobs:
  deploy:
    runs-on: ubuntu-latest
    env:
      DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
      DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - name: Deploy to prod
        run: databricks bundle deploy -t prod
"""
print(yaml_cd)

### 💻 Na prática — Testes no CI

Adicione um step que roda o notebook como teste (ex.: pytest sobre funções de transformação extraídas em `.py`).


In [ ]:
# Boa prática: extrair lógica para .py e testar com pytest
codigo = """
def filtra_e_enriquece(df):
    return df.filter(col("Quantity") > 0).withColumn("receita", col("Quantity") * col("UnitPrice"))

def test_filtra_e_enriquece(spark):
    df = spark.createDataFrame([(1, 10.0), (-1, 5.0)], ["Quantity", "UnitPrice"])
    out = filtra_e_enriquece(df)
    assert out.count() == 1
    assert out.collect()[0]["receita"] == 10.0
"""
print(codigo)

> 🎯 **Dica de prova**: A DEP cobra o papel do CI/CD e os blueprints oficiais. Memorize: CI valida em PR; CD deploya em merge; secrets via GitHub Secrets.


## 🎯 Exercícios de fixação

**1.** Crie o workflow de CI no seu repo e rode um PR de teste.

**2.** O que é um secret no GitHub Actions e onde configurá-lo?

**3.** Por que CI roda validate e não deploy direto?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** CI no repo

Push do .github/workflows/ci.yml → abrir PR → ver o check 'validate' verde → merge.

**2.** Secrets

Settings → Secrets and variables → Actions: DATABRICKS_HOST e DATABRICKS_TOKEN ficam criptografados; o workflow usa ${{ secrets.X }}.

**3.** validate não deploy

Deploy em PR criaria recursos de staging/prod a cada commit — caro e arriscado. CI valida (rápido, sem efeito); CD deploya só no merge.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*